In [ ]:
# test.py
from __future__ import annotations

import	 yaml
from pathlib import Path
from dotenv import load_dotenv

from IPython.display import Image, display

from src.core.state import InputState, Context
from src.graph.workflow import build_graph

load_dotenv()

In [ ]:
def get_config(config_dir: str = "src/core") -> dict:
    """Load model, path, and directory settings into a single config mapping.

    The application keeps runtime settings in separate YAML files, so this helper
    merges them into one dictionary that the workflow can consume consistently.
    """
    config_dir_path = Path(config_dir)

    with open(config_dir_path / "config.yaml", "r") as file:
        config_data = yaml.safe_load(file)
    with open(config_dir_path / "paths.yaml", "r") as file:
        paths_data = yaml.safe_load(file)

    return {**config_data, **paths_data}

In [ ]:
print("Loading application configuration...\n")
app_config = get_config()

global_settings = app_config["global"]
extractor_settings = app_config["agents"]["extractor"]
base_dirs = app_config["base_directories"]
input_dirs = app_config["inputs"]
output_dirs = app_config["outputs"]

print("Building workflow graph...\n")
graph = build_graph(
    model=global_settings["default_model"],
    ocr_model=extractor_settings["ocr_model"],
    temperature=global_settings["default_temperature"],
    max_tokens=global_settings["max_tokens"]
)

# display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
sheets = [sheet.stem for sheet in Path("data/output").glob('sheet_*')]

for sheet in sheets[20:40]:
    print(f"Processing {sheet}:\n")
    print("Invoking graph with test data...\n")
    final_state = graph.invoke(
        input=InputState({
            "sheet_id": sheet,
            "dpi": 300,
            "max_regrade": 2,
            "do_extract": False,
            "do_feedback": False,
            "exam_file": "exam_java.yaml",
            "rubric_file": "java_criteria.yaml",
            "review_criteria_file": "review_criteria.yaml"
        }),
        context=Context(
            sheets_dir=Path(input_dirs["sheets_dir"]),
            exams_dir=Path(input_dirs["exams_dir"]),
            criteria_dir=Path(input_dirs["criteria_dir"]),
            output_base=Path(base_dirs["output_base"]),
            export_dir=Path(output_dirs["exports_dir"])
        )
    )

print(f"Processed {len(sheets[20:40])} sheets")

# Tests

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, cohen_kappa_score
from src.utils import io

### Data

In [ ]:
results = pd.read_excel("data/output/exports/results.xlsx")
results.drop_duplicates(subset=['sheet_id'], keep='last', inplace=True)
results.rename(columns={'total_mark': 'ai_mark', 'Q1': 'q1_ai_mark', 'Q2': 'q2_ai_mark', 'Q3': 'q3_ai_mark', 'Q4': 'q4_ai_mark'}, inplace=True)

human_grades = pd.read_csv("data/input/human_grading.csv")
human_grades.rename(columns={'total_mark': 'human_mark', 'Q1': 'q1_human_mark', 'Q2': 'q2_human_mark', 'Q3': 'q3_human_mark', 'Q4': 'q4_human_mark'}, inplace=True)

tool_results = pd.read_excel("data/output/exports/tools.xlsx")
tool_results.drop_duplicates(subset=['sheet_id'], keep='last', inplace=True)

df = pd.merge(results, human_grades, on='sheet_id', how='inner')
df = pd.merge(df, tool_results, on='sheet_id', how='inner')

# We bin the continuous grades into standard letter grades.
bins = [0, 19, 39, 59, 79, 100]
labels = ['F', 'D', 'C', 'B', 'A']
df['ai_letter'] = pd.cut(df['ai_mark'], bins=bins, labels=labels, include_lowest=True)
df['human_letter'] = pd.cut(df['human_mark'], bins=bins, labels=labels, include_lowest=True)

# Calculate the absolute error for every row (used in secondary metrics)
df['absolute_error'] = np.abs(df['human_mark'] - df['ai_mark'])
# for q in range(1, 5):
#     df[f'q{q}_absolute_error'] = np.abs(df[f'q{q}_human_mark'] - df[f'q{q}_ai_mark'])

# Set academic plotting style
sns.set_theme(style="whitegrid")

# Create a dictionary to store the analysis results
analysis_json = {}

print(df)

### 1. Primary Metrics

In [ ]:
# 1.1 Mean Absolute Error (MAE)

mae = mean_absolute_error(df['human_mark'], df['ai_mark'])
analysis_json['mean_absolute_error'] = round(mae, 2)
print(f"Mean Absolute Error (MAE): {mae:.2f} marks")

In [ ]:
# 1.2 Root Mean Square Error (RMSE)

rmse = np.sqrt(mean_squared_error(df['human_mark'], df['ai_mark']))
analysis_json['root_mean_squared_error'] = round(rmse, 2)
print(f"Root Mean Square Error (RMSE): {rmse:.2f} marks")

In [ ]:
# 1.3 Grade Agreement

# Defining agreement as the AI being within a strict tolerance (e.g., ± 5 marks)
df['exact_match'] = df['human_mark'] == df['ai_mark']
exact_match = (df['exact_match'].mean()) * 100
df['within_3_points'] = df['absolute_error'] <= 3
within_3_points = (df['within_3_points'].mean()) * 100
df['within_5_points'] = df['absolute_error'] <= 5
within_5_points = (df['within_5_points'].mean()) * 100
df['within_10_points'] = df['absolute_error'] <= 10
within_10_points = (df['within_10_points'].mean()) * 100
analysis_json['exact_match'] = f"{exact_match:.2f}%"
analysis_json['within_3_points'] = f"{within_3_points:.2f}%"
analysis_json['within_5_points'] = f"{within_5_points:.2f}%"
analysis_json['within_10_points'] = f"{within_10_points:.2f}%"
analysis_json['grade_agreement(±5)'] = f"{within_5_points:.2f}%"
print(f"Exact Match: {exact_match:.2f}%")
print(f"Within 3 Points: {within_3_points:.2f}%")
print(f"Within 5 Points: {within_5_points:.2f}%")	
print(f"Within 10 Points: {within_10_points:.2f}%")
print(f"Grade Agreement (±{5} marks): {within_5_points:.2f}%")

In [ ]:
# Identity Scatter Plot

axes = sns.scatterplot(data=df, x='human_mark', y='ai_mark', alpha=0.7)
axes.plot([0, 100], [0, 100], color='red', linestyle='--', label='Line of Perfect Equality ($Y=X$)')
minimum = min(df['human_mark'].min(), df['ai_mark'].min()) - 5
axes.set_xlim((minimum//10)*10, 100)
axes.set_ylim((minimum//10)*10, 100)
axes.set_title('AI vs. Human Grade Alignment')
axes.set_xlabel('Historical Human Benchmark Grade')
axes.set_ylabel('Multi-Agent Pipeline Predicted Grade')

plt.show()

In [ ]:
# 1.4 Cohen's Kappa (Inter-rater reliability)

# Cohen's Kappa works best with categorical/ordinal data. 
# We use quadratic weights because the grades are ordinal (e.g., confusing an A for a B is 
# less severe than confusing an A for an F).
kappa = cohen_kappa_score(df['human_letter'], df['ai_letter'], weights='quadratic')
analysis_json['cohen_kappa'] = round(kappa, 2)
print(f"Cohen's Kappa (Quadratic Weighted): {kappa:.3f}")

In [ ]:
# Confusion Matrix Heatmap
letter_order = []
for letter in labels:
	if letter in df['human_letter'].values or letter in df['ai_letter'].values:
		letter_order.append(letter)
conf_matrix = pd.crosstab(df['human_letter'], df['ai_letter'], rownames=['Human'], colnames=['AI']).reindex(index=letter_order, columns=letter_order, fill_value=0)
axes = sns.heatmap(data=conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False)
axes.set_title("Letter Grade Confusion Matrix (Basis for Cohen's Kappa)")

plt.show()

### 2. Secondary Metrics

In [ ]:
# # 2.1 Variance in grading across Questions

# print("1. Performance Variability by Question (Mean Error & Variance):")
# stats_by_qs = pd.DataFrame({'question': range(1, 5)})
# for q in range(5):
# 	stats_by_qs.loc[q, 'mae'] = df[f'q{q+1}_absolute_error'].mean()
# 	stats_by_qs.loc[q, 'error variance'] = df[f'q{q+1}_absolute_error'].var()

# print(stats_by_qs)

In [ ]:
# 2.2 Scoring consistency across grade ranges

# This checks for proportional bias (e.g., is the AI harsher on failing students vs. top students?)
print("3. Scoring Consistency Across Grade Ranges:")
# Group by the human's letter grade to see where the AI struggles the most
grade_range_stats = df.groupby('human_letter', observed=False)['absolute_error'].agg(['mean', 'var', 'count']).round(2)
grade_range_stats.rename(columns={'mean': 'MAE', 'var': 'Error Variance', 'count': 'Sample Size'}, inplace=True)
analysis_json['grade_range_stats'] = grade_range_stats.to_dict(orient='index')
print(grade_range_stats)


In [ ]:
# Box Plots for Error Variance across Questions
# fig, axes = plt.subplots(1, 2, figsize=(16, 12))
# q_data = {
# 	'question': np.repeat(range(1, 5), len(df)),
# 	'absolute_error': df[[f'q{i}_absolute_error' for i in range(1, 5)]].to_numpy().flatten(order='F')
# }
# sns.boxplot(ax=axes[0], data=q_data, x='question', y='absolute_error')
# axes[0].set_title('Absolute Error Distribution by Question')
# axes[0].set_xlabel('Question Number')
# axes[0].set_ylabel('Absolute Error Magnitude (|Human - AI|)')

# Box Plots for Error Variance across Grade Ranges
df["human_letter"] = pd.Categorical(df["human_letter"], categories=letter_order, ordered=True)
axes = sns.boxplot(data=df, x='human_letter', y='absolute_error')
axes.set_title('Absolute Error Distribution by Human Letter Grade')
axes.set_xlabel('Human Letter Grade')
axes.set_ylabel('Absolute Error Magnitude (|Human - AI|)')

plt.show()

### 3. Tools Analysis

In [ ]:
# 1. Average tool calls

average_tool_calls = np.floor(df["tools_total"].mean())
analysis_json['average_tool_calls'] = average_tool_calls
print(f"Average tool calls: {average_tool_calls}")

In [ ]:
# Safe the analysis results to a JSON file for further review or reporting
io.write_json("data/output/exports/analysis_results.json", analysis_json)